# Lab 1A — The Modern GenAI Stack
**Day 1 Morning | ~45 minutes | Colab CPU | OpenAI API key required**

Lab 0 proved the libraries load. This notebook puts them to work.

## What you will build

1. Run local **GPT-2** through HuggingFace `pipeline()`, then open it: tokens → logits → next-token probabilities
2. Call hosted **gpt-5-mini** with the OpenAI client (the same pattern every later lab uses)
3. Stream tokens as they arrive
4. Swap **one string** — `base_url` — so the same client can point at a different backend
5. Wrap that client in a tiny LangChain chain (template → model → parser)

> **The key idea:** The OpenAI Chat Completions format is the wire protocol of this course.
> Change `base_url`. Keep everything else. Lab 5 is the same idea with *your* FastAPI server.

```
Part A (local)              Part B (hosted API)              Part C (framework)
GPT-2 on CPU           →    gpt-5-mini + base_url      →    LangChain on the same client
tokens, logits, memory      stream, quality, swap            template | llm | parser
```

**Coming from Lab 0:** you already turned a sentence into GPT-2 token IDs. Part A puts a **model** on those IDs. Then we leave the 124M toy and talk to a hosted model the way production apps do.


---

## 0. Install and configure

**Why:** Colab starts empty. We need Transformers (local GPT-2), the OpenAI SDK (hosted calls), and LangChain (Part C).

**What:** The same three-line installer as Lab 0. Then a config cell that reads `OPENAI_API_KEY` from Colab Secrets when you are in Colab, or from the repo's `.env` file when you run locally. Either way the key lands in an environment variable and never in a cell.

**When:** Once per new runtime.

In [ ]:
import sys
%pip install -q uv
!uv pip install -q --python {sys.executable} "transformers>=5" torch openai langchain langchain-openai langchain-core httpx python-dotenv

In [ ]:
import os
try:
    from google.colab import userdata          # Colab: read the Secret you added
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv             # local: read .env in the repo root
    load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "Add OPENAI_API_KEY as a Colab Secret or to .env"

OPENAI_API_KEY  = os.environ["OPENAI_API_KEY"]
OPENAI_BASE_URL = "https://api.openai.com/v1"
DEFAULT_MODEL   = "gpt-5-mini"   # workhorse for this course
QUALITY_MODEL   = "gpt-5"        # quality / judge comparisons only
print(f"Ready — {DEFAULT_MODEL} at {OPENAI_BASE_URL}, key starts {OPENAI_API_KEY[:8]}...")

**Checkpoint:** you should see `gpt-5-mini` and a key prefix, not the full key. If you get `Missing OPENAI_API_KEY`, the secret name is wrong or Notebook access is off.

---

## HuggingFace API tiers

HuggingFace gives you several levels of control. Part A walks **down** the stack so you can pick the right one later.

| Level | API | Best for | What you trade away |
| --- | --- | --- | --- |
| High-level | `pipeline()` | Fast prototypes | Less control over tokens and internals |
| Mid-level | `AutoTokenizer` + `AutoModelForCausalLM` | Custom prompts, generation settings, inspection | More boilerplate |
| Low-level | Direct forward pass / custom `generate()` | Research, debugging a serving bug | You own more details |
| Serving | `transformers serve` | Local OpenAI-compatible endpoint | Not vLLM/TGI at scale |

Transformers **v5** (what we install) loads quantized models with `quantization_config` — not the old `load_in_4bit=True` shortcut. Lab 4 uses that modern pattern.


---

## Part A — HuggingFace from high-level to low-level (~15 min)

We start local and small: **GPT-2, 124M parameters, Colab CPU**. The point is not quality. The point is to **open the black box** you will later serve, quantize, and wrap in RAG.

You will see three views of the same model:

1. `pipeline()` — task-oriented inference in a few lines
2. `AutoTokenizer` + `generate()` — controlled generation
3. A raw forward pass — logits and next-token probabilities

> If you see a warning about `h.{0...11}.attn.bias` marked UNEXPECTED, ignore it. Those are causal-mask buffers from the original OpenAI GPT-2 checkpoint. If the model prints coherent text, it loaded correctly.


In [ ]:
import transformers
print(f"Transformers version: {transformers.__version__}")
print("v5 note: AutoTokenizer picks the tokenizer backend for you.")


### High-level: `pipeline()`

**Why:** This is the fastest way to go from a Hub model id to text. Production serving rarely stays here, but it is how you smoke-test a checkpoint.

**What:** Load GPT-2, ask it to continue a deployment sentence, cap it at 40 new tokens.

**When:** Demos, first look at a new model, "does this checkpoint run at all?"


In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")
result = generator(
    "The best way to deploy a language model is",
    max_new_tokens=40,
)
print(result[0]["generated_text"])

In [ ]:
from transformers import pipeline, GenerationConfig

generator = pipeline("text-generation", model="gpt2")
gen_config = GenerationConfig(max_new_tokens=40)

result = generator(
    "The best way to deploy a language model is",
    generation_config=gen_config,
)
print(result[0]["generated_text"])


**Checkpoint:** you get a continuation. It will sound 2019-era and a bit rambling. That is GPT-2. We are here for the machinery, not the prose.

---

### Mid-level: `AutoTokenizer` + `AutoModelForCausalLM`

`pipeline()` hid three steps: tokenize, run the model, decode. Production code usually wants those steps named — so you can log token counts, set `max_new_tokens`, or inspect failures.

`Auto*` reads the checkpoint metadata and loads the right classes. You do not import `GPT2LMHeadModel` by hand.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

inputs = tokenizer("The best way to deploy", return_tensors="pt")
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))


### Unwrap step 1 — text → token IDs

Lab 0 did this on a greeting. Same tokenizer, now as the **actual input** to a model. Every later API call (OpenAI, Groq, your Lab 5 server) is still "text in, token IDs inside, text out."


In [ ]:
text = "The best way to deploy a language model is"
inputs = tokenizer(text, return_tensors="pt")
token_ids = inputs["input_ids"][0]

print("STEP 1 — TEXT → TOKENS")
print(f"  Input text : {text!r}")
print(f"  Token IDs  : {token_ids.tolist()}")
print(f"  Pieces     : {[tokenizer.decode([t]) for t in token_ids]}")
print(f"  Count      : {len(token_ids)} tokens")


### Unwrap step 2 — token IDs → logits → next-token probabilities

**Why:** Generation is not a paragraph appearing at once. The model scores **every** vocab item for the *next* token, then samples. Temperature (Lab 3) only makes sense once you have seen this vector.

**What:** One forward pass. Softmax. Print the top 5 candidates.


In [ ]:
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits                      # [batch, sequence, vocab]
next_token_logits = logits[0, -1, :]         # scores for the next token
probs = torch.softmax(next_token_logits, dim=-1)
top5 = torch.topk(probs, 5)

print("STEP 2 — MODEL PREDICTS NEXT TOKEN")
print(f"  Vocab size : {logits.shape[-1]:,}")
print()
print("  Top 5 candidates:")
for score, idx in zip(top5.values, top5.indices):
    piece = tokenizer.decode([idx.item()])
    print(f"    {piece!r:<20} {score.item() * 100:.1f}%")


**Checkpoint:** the percentages sum to less than 100% (we only printed five). The rest of the mass is spread over ~50,000 other tokens. Sampling picks from this distribution. That is all "the model wrote a sentence" ever was.

---

### Memory math — why size is a deployment decision

**Why:** Lab 4 will quantize a real model. Before that, you need the arithmetic: parameters × bytes per parameter ≈ weight memory. KV cache and activations are extra; this table is the floor.

**When:** Any time someone says "just host the 70B."


In [ ]:
def memory_table(model, label):
    params = sum(p.numel() for p in model.parameters())
    print(label)
    print(f"  Parameters : {params:,}  ({params / 1e6:.0f}M)")
    print(f"  FP32       : {params * 4 / 1e6:.0f} MB")
    print(f"  FP16       : {params * 2 / 1e6:.0f} MB")
    print(f"  INT4       : {params * 0.5 / 1e6:.0f} MB")
    print("  Scale-up   → a 7B model is ~14 GB in FP16, ~3.5 GB in INT4")

memory_table(model, "GPT-2 (124M params)")


### How you actually load INT4 (Lab 4)

The table is arithmetic. Loading quantized weights uses a config object in Transformers v5:

```python
from transformers import BitsAndBytesConfig

qconfig = BitsAndBytesConfig(load_in_4bit=True)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=qconfig,
    device_map="auto",
)
```

Older blog posts pass `load_in_4bit=True` straight into `from_pretrained()`. Treat those as legacy. We are not running this here — GPT-2 on CPU does not need it, and bitsandbytes wants a GPU. Lab 4 runs it on a T4.

### Serving bridge: `transformers serve`

Transformers v5 ships a small OpenAI-compatible server. This is the same `base_url` idea as Part B, pointed at a local process. Optional, not required for class:

```bash
pip install "transformers[serving]>=5"
transformers serve --force-model Qwen/Qwen2.5-0.5B-Instruct --port 8000
```

```python
client = OpenAI(api_key="not-needed", base_url="http://localhost:8000/v1")
```

For high-scale production, the course still points at **vLLM** (Lab 5, concepts only). `transformers serve` is the classroom-sized bridge.


---

## Part B — Hosted LLM via the OpenAI client (~15 min)

Local GPT-2 showed the machinery. Production apps usually call a **server**. The server might be OpenAI, Groq, vLLM, Ollama, or the FastAPI proxy you will build in **Lab 5**.

> **The key pattern** — one client, swappable backends:
>
> ```python
> client = OpenAI(api_key=YOUR_KEY, base_url=YOUR_URL)  # this line is the whole course
> ```

### The Chat Completions wire format

The Python SDK is a wrapper around HTTP. The request body is a list of `messages`:

| Role | Job |
|------|-----|
| `system` | High-priority behavior ("be concise", "only use the tools") |
| `user` | The request |
| `assistant` | Previous model turns (Lab 3 chat history; Lab 1B tool calls) |
| `tool` | Tool results (Part 2 of this lab) |

Order matters. A chat request is a transcript, not one prompt string.

In the response, watch:

- `choices[0].message.content` — the text
- `choices[0].finish_reason` — `stop`, `length`, `tool_calls`, …
- `usage` — token accounting (this is what you pay for)


In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY, base_url=OPENAI_BASE_URL)

messages = [
    {"role": "system", "content": "You are an LLM deployment expert. Be concise."},
    {"role": "user", "content": "What are the top 3 reasons LLM demos fail in production?"},
]

response = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=messages,
)

print(response.choices[0].message.content)
print()
print("finish_reason:", response.choices[0].finish_reason)
print("tokens used  :", response.usage.total_tokens)


### Inspect the raw object

Before adding frameworks, look at the protocol. Your Lab 5 FastAPI server will have to return these same fields.


In [ ]:
raw = response.model_dump()

print("Top-level fields:", list(raw.keys()))
print("id              :", raw["id"])
print("model           :", raw["model"])
print("finish_reason   :", raw["choices"][0]["finish_reason"])
print("message role    :", raw["choices"][0]["message"]["role"])
print("usage           :", raw["usage"])


### Streaming

**Why:** Users hate waiting for a full paragraph. Serving stacks (vLLM, your Lab 5 SSE endpoint, Gradio in Lab 7) all stream.

**What:** `stream=True` yields chunks. Each chunk has a `delta.content` that may be `None` — skip those.

**When:** Chat UIs, long answers, anything a human is watching.


In [ ]:
print("Streaming (tokens appear as they are generated):")
print("-" * 60)

stream = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[
        {
            "role": "user",
            "content": "Explain the difference between FP16 and INT4 in 4 short bullet points.",
        }
    ],
    stream=True,
)
for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end="", flush=True)

print()
print("-" * 60)


### Same client, different model id

Change only the `model` string. The client does not care. `gpt-5-mini` is the workhorse for the rest of the course. `gpt-5` is the quality / judge model (Lab 6 RAGAS).


In [ ]:
question = "In one sentence: what is the most important thing to know about LLM serving?"

r_mini = client.chat.completions.create(
    model=DEFAULT_MODEL,
    messages=[{"role": "user", "content": question}],
)
r_full = client.chat.completions.create(
    model=QUALITY_MODEL,
    messages=[{"role": "user", "content": question}],
)

print("gpt-5-mini:", r_mini.choices[0].message.content)
print("gpt-5     :", r_full.choices[0].message.content)


### THE KEY MOMENT — swap `base_url`, keep everything else

This also works with:

| Backend | `base_url` | Key |
|---------|------------|-----|
| OpenAI | `https://api.openai.com/v1` | `OPENAI_API_KEY` |
| Groq | `https://api.groq.com/openai/v1` | `GROQ_API_KEY` (optional secret) |
| vLLM / Ollama | `http://your-gpu:8000/v1` | often unused |
| Lab 5 FastAPI + ngrok | `https://<ngrok>/v1` | unused |

If the instructor added a Colab Secret named `GROQ_API_KEY`, the next cell calls Groq with the **same** `OpenAI(...)` constructor. If not, you still see the OpenAI row — the pattern is the point, not Groq itself.


In [ ]:
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get("GROQ_API_KEY")
except Exception:                      # not on Colab, or no such secret
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

providers = {
    "OpenAI gpt-5-mini": {"base_url": OPENAI_BASE_URL, "api_key": OPENAI_API_KEY, "model": DEFAULT_MODEL},
}
if GROQ_API_KEY:
    providers["Groq llama-3.1-8b-instant"] = {
        "base_url": "https://api.groq.com/openai/v1", "api_key": GROQ_API_KEY, "model": "llama-3.1-8b-instant",
    }
else:
    print("No GROQ_API_KEY — showing OpenAI only. A second provider is one more dict entry.\n")

q = "In one sentence: what is PagedAttention?"
for name, cfg in providers.items():
    c = OpenAI(api_key=cfg["api_key"], base_url=cfg["base_url"])
    r = c.chat.completions.create(model=cfg["model"], messages=[{"role": "user", "content": q}])
    print(f"[{name}]\n  {r.choices[0].message.content}\n")

**Checkpoint:** you called a hosted model, streamed it, compared two model ids, and (optionally) two providers, without changing the request shape. That is the deployment idea of the course.

---

## Part C — LangChain on the same backend (~15 min)

So far you called the API directly. That is the right baseline. Frameworks help when the app grows: prompt templates, parsers, retries, tracing. They do **not** serve the model. They call the same OpenAI-compatible backend.

The goal is not to memorize LangChain. The goal is: same `base_url`, higher-level app structure.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    base_url=OPENAI_BASE_URL,
    model=DEFAULT_MODEL,
)

lc_messages = [
    SystemMessage(content="You are an LLM deployment expert. Be concise."),
    HumanMessage(
        content="What is the difference between vLLM and a simple FastAPI proxy for LLM inference?"
    ),
]
print(llm.invoke(lc_messages).content)


### Templates + the pipe operator

`ChatPromptTemplate` fills `{placeholders}`. `StrOutputParser` turns a chat message into a plain string. The `|` operator is LCEL: prompt → model → parser. Swap any piece later without rewriting the others.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a technical instructor. Explain with one analogy."),
        ("user", "Explain {concept} to someone who {background}."),
    ]
)

chain = template | llm | StrOutputParser()

print(
    chain.invoke(
        {
            "concept": "quantization",
            "background": "has never worked with neural networks",
        }
    )
)


---

## Conceptual checkpoint

You saw the same deployment story at three layers:

1. **Local model** — `pipeline()` hides tokenize/generate/decode; you opened it.
2. **Hosted API** — Chat Completions sends ordered `messages` to an OpenAI-compatible URL.
3. **Framework** — LangChain wraps that backend with templates and parsers.

Part 2 adds one capability: instead of only returning text, the model can **request that your code run a tool**.

## Lab 1A complete

- [ ] GPT-2 token IDs and top-5 next-token probabilities
- [ ] Memory table (FP32 / FP16 / INT4)
- [ ] A streaming response from gpt-5-mini
- [ ] Two-model quality comparison
- [ ] LangChain chain producing an analogy

## Stretch (optional)

1. Call the same prompt at temperature `0`, `0.5`, and `1.0`. What changes? What stays the same?
2. Browse `huggingface.co/models?pipeline_tag=text-generation`. Find a SQL-tuned model and note its size.
3. Add `logprobs=True` to a non-streaming OpenAI call. Print the top tokens for the first word.
4. Add a `GROQ_API_KEY` secret and re-run the provider cell. Compare latency vs gpt-5-mini.
5. Memory math: 7B, 13B, 70B at FP16 and INT4. Which fits a T4 (16 GB) for *weights only*?
6. After class: `transformers serve --force-model Qwen/Qwen2.5-0.5B-Instruct --port 8000`, then point `OPENAI_BASE_URL` at `http://localhost:8000/v1`.

## Next: Lab 1B

Part 1 treated the LLM as an inference engine: prompt in, text out. Part 2 changes the mental model. The LLM can request actions through tools, observe the result, and then answer with grounded data.
